In [ ]:
import os
from os.path import expanduser
home = expanduser("~/")

import sys
# sys.path.insert(0, '/global/u2/x/xshuang/gigalens-xh-dev/src')

# import sys
conda_env = sys.path[1]
del sys.path[1]

import os
# sys.path.append(f'{os.environ['HOME']}/gigalens_personal/gigalens/src')
sys.path.append(home+'/gigalens'+'/src')
sys.path.append(conda_env)
sys.path.append(home+'/GIGALens-Code/')
print(sys.path)



srcdir = os.path.join(home, "gigalens/src/")


In [ ]:
import tensorflow_probability.substrates.jax as tfp

from gigalens.jax.inference import ModellingSequence
from gigalens.jax.model import ForwardProbModel, BackwardProbModel
from gigalens.model import PhysicalModel
from gigalens.jax.simulator import LensSimulator
from gigalens.simulator import SimulatorConfig
from gigalens.jax.profiles.light import sersic
from gigalens.jax.profiles.mass import epl, shear

import jax
from jax import random
import numpy as np
import optax
from jax import numpy as jnp
from matplotlib import pyplot as plt
import optax
import corner
import yaml
import pickle
from helpers import *
import blackjax
import importlib
tfd = tfp.distributions

In [ ]:
prior = make_default_prior()
gigal_dir = os.path.join(home,'gigalens/src/gigalens/')
kernel = np.load(gigal_dir + '/assets/psf.npy').astype(np.float32)
sim_config = SimulatorConfig(delta_pix=0.065, num_pix=60, supersample=2, kernel=kernel)
phys_model = PhysicalModel([epl.EPL(50), shear.Shear()], [sersic.SersicEllipse(use_lstsq=False)], [sersic.SersicEllipse(use_lstsq=False)])
lens_sim = LensSimulator(phys_model, sim_config, bs=1)
observed_img = np.load(gigal_dir + '/assets/demo.npy')
prob_model = ForwardProbModel(prior, observed_img, background_rms=0.2, exp_time=100)
model_seq = ModellingSequence(phys_model, prob_model, sim_config)

results = {}
results["MAP"] = MAPResults.load(os.path.join(home, "GIGALens-Code/alternate_inference/test_system"), model_seq)
results["SVI"] = SVIResults.load(os.path.join(home, "GIGALens-Code/alternate_inference/test_system"), model_seq)
results["HMC"] = HMCResults.load(os.path.join(home, "GIGALens-Code/alternate_inference/test_system"), model_seq)

In [ ]:
# prior = make_default_prior()
# kernel = np.load('/global/homes/l/linusu/gigalens/src/gigalens/assets/psf.npy').astype(np.float32)
# sim_config = SimulatorConfig(delta_pix=0.065, num_pix=80, supersample=2, kernel=kernel)
# phys_model = PhysicalModel([epl.EPL(50), shear.Shear()], [sersic.SersicEllipse(use_lstsq=False)], [sersic.SersicEllipse(use_lstsq=False)])
# lens_sim = LensSimulator(phys_model, sim_config, bs=1)

# systems_dir = os.path.join(home, "GIGALens-Code", "SystemSaves")
# f = np.load(os.path.join(systems_dir, "100SystemsStandard80px.npz"))
# keys = f.files
# observed_imgs = jnp.array([f[key] for key in keys])
# observed_img = observed_imgs[60]

# prob_model = ForwardProbModel(prior, observed_img, background_rms=0.2, exp_time=100)
# model_seq = ModellingSequence(phys_model, prob_model, sim_config)

# results_dir = os.path.join(home, "sys60_converged_4e4_burnin")
# results = {}
# results["MAP"] = MAPResults.load(results_dir, model_seq)
# results["SVI"] = SVIResults.load(results_dir, model_seq)
# results["HMC"] = HMCResults.load(results_dir, model_seq)

In [ ]:
lens_prior = tfd.JointDistributionSequential(
    [
        tfd.JointDistributionNamed(
            dict(
                theta_E=tfd.LogNormal(jnp.log(1.25), 0.4),
                gamma=tfd.TruncatedNormal(2, 0.5, 1, 3),
                e1=tfd.Normal(0, 0.2),
                e2=tfd.Normal(0, 0.2),
                center_x=tfd.Normal(0, 0.06),
                center_y=tfd.Normal(0, 0.06),
            )
        ),
        tfd.JointDistributionNamed(
            dict(gamma1=tfd.Normal(0, 0.1), gamma2=tfd.Normal(0, 0.1))
        ),
    ]
)
lens_light_prior = tfd.JointDistributionSequential(
    [
        tfd.JointDistributionNamed(
            dict(
                R_sersic=tfd.LogNormal(jnp.log(1.6), 0.25),
                n_sersic=tfd.Uniform(0.5, 8),
                e1=tfd.TruncatedNormal(0, 0.1, -0.15, 0.15),
                e2=tfd.TruncatedNormal(0, 0.1, -0.15, 0.15),
                center_x=tfd.Normal(0, 0.02),
                center_y=tfd.Normal(0, 0.02),
                Ie=tfd.LogNormal(jnp.log(300.0), 0.5),
            )
        )
    ]
)

source_light_prior = tfd.JointDistributionSequential(
    [
        tfd.JointDistributionNamed(
            dict(
                R_sersic=tfd.LogNormal(jnp.log(0.25), 0.25),
                n_sersic=tfd.Uniform(0.5, 8),
                e1=tfd.TruncatedNormal(0, 0.3, -0.5, 0.5),
                e2=tfd.TruncatedNormal(0, 0.3, -0.5, 0.5),
                center_x=tfd.Normal(0, 0.5),
                center_y=tfd.Normal(0, 0.5),
                Ie=tfd.LogNormal(jnp.log(150.0), 0.9),
            )
        ),
        tfd.JointDistributionNamed(
            dict(
                R_sersic=tfd.LogNormal(jnp.log(0.25), 0.25),
                n_sersic=tfd.Uniform(0.5, 8),
                e1=tfd.TruncatedNormal(0, 0.3, -0.5, 0.5),
                e2=tfd.TruncatedNormal(0, 0.3, -0.5, 0.5),
                center_x=tfd.Normal(0, 0.5),
                center_y=tfd.Normal(0, 0.5),
                Ie=tfd.LogNormal(jnp.log(150.0), 0.9),
            )
        )

    ]
)

prior = tfd.JointDistributionSequential(
    [lens_prior, lens_light_prior, source_light_prior]
)

In [ ]:
# i = 79
# # prior = make_default_prior()
# kernel = np.load('/global/homes/l/linusu/gigalens/src/gigalens/assets/psf.npy').astype(np.float32)
# sim_config = SimulatorConfig(delta_pix=0.065, num_pix=80, supersample=2, kernel=kernel)
# phys_model = PhysicalModel([epl.EPL(50), shear.Shear()], [sersic.SersicEllipse(use_lstsq=False)], [sersic.SersicEllipse(use_lstsq=False), sersic.SersicEllipse(use_lstsq=False)])
# lens_sim = LensSimulator(phys_model, sim_config, bs=1)

# systems_dir = os.path.join(home, "GIGALens-Code", "SystemSaves")
# f = np.load(os.path.join(systems_dir, "100SystemsStandard80px.npz"))
# keys = f.files
# observed_imgs = jnp.array([f[key] for key in keys])
# observed_img = observed_imgs[i]

# prob_model = ForwardProbModel(prior, observed_img, background_rms=0.2, exp_time=100)
# model_seq = ModellingSequence(phys_model, prob_model, sim_config)

# normal_results_loc = os.path.join(home, f"GIGALens-Code/pipeline_results/100standard80px")
# results_dir = os.path.join(home, normal_results_loc, f"{i}")

# # results = {}
# # results["MAP"] = MAPResults.load(results_dir, model_seq)
# # results["SVI"] = SVIResults.load(results_dir, model_seq)
# # results["HMC"] = HMCResults.load(results_dir, model_seq)

In [ ]:
# results = {}
# map_opt = optax.adabelief(1e-2, b1=0.95, b2=0.99, nesterov=True)
# best, lp, chisq = model_seq.MAP(map_opt, n_samples=500, num_steps=500)


In [ ]:
# svi_opt = optax.adabelief(1e-4, b1=0.95, b2=0.99)
# qz, loss_hist = model_seq.SVI(best, svi_opt, num_steps=500, n_vi=250)


In [ ]:
# hmc_samples = model_seq.HMC(qz)
# hmc_samples = hmc_samples.transpose((1, 2, 0, 3))
# hmc_samples = hmc_samples.reshape(hmc_samples.shape[0]*hmc_samples.shape[1], *hmc_samples.shape[2:])

In [ ]:
# qz = results["SVI"].qz
best = results["MAP"].best_z
default_start = jnp.diag(jnp.ones((best.shape[-1],))) * 1e-3
no_SVI_qz = tfd.MultivariateNormalTriL(loc=jnp.squeeze(best), scale_tril=default_start)
qz = no_SVI_qz

hmc_samples = results["HMC"].HMC_samples_z#.reshape(64, 250000, 22)
# hmc_samples = hmc_samples.transpose((1, 2, 0, 3))
hmc_samples = hmc_samples.reshape(hmc_samples.shape[0]*hmc_samples.shape[1], *hmc_samples.shape[2:])

In [ ]:
hmc_samples.shape

In [ ]:
import mclmc_alt
importlib.reload(mclmc_alt)
from mclmc_alt import MCLMC_JIT

num_burnin_steps = 2000
num_results=2000
frac_tune1=0.2 #* initial step size tuning
frac_tune2=0.6 #* Used for mass matrix adaptation
frac_tune3=0.2 #! Tuning L. ~10 effective samples are needed for this to be accurate

debug_hist_mclmc = MCLMC_JIT(
    model_seq, qz, 
    n_hmc=16, num_burnin_steps=num_burnin_steps, num_results=num_results, 
    desired_energy_variance=5e-4, frac_tune1=frac_tune1, frac_tune2=frac_tune2, frac_tune3=frac_tune3,
    seed=0, debug_output=True, step_size_adapt_use_psmile=False, use_shard_map=True,
)

In [ ]:
debug_hist = debug_hist_mclmc
mclmc_samples = debug_hist_mclmc.position[:, -num_results:, :]
dim = mclmc_samples.shape[-1]
samples = mclmc_samples[:, :,:].reshape(-1, dim)

final_cov = jnp.cov(samples.T)
# final_mean = jnp.mean(samples, axis=0)
# qz_true = tfd.MultivariateNormalFullCovariance(loc=final_mean, covariance_matrix=final_cov)

In [ ]:
stage1 = int(frac_tune1*num_burnin_steps)
stage2 = int((frac_tune1+frac_tune2)*num_burnin_steps)
stage3 = int((frac_tune1+frac_tune2+frac_tune3)*num_burnin_steps)

fig, axs = plt.subplots(5, 1, sharex=True)
ax1, ax2, ax3, ax4, ax_last =axs
fig.set_size_inches(10, 8)
ax1.plot(debug_hist.step_size.T)
ax1.set_title("Chain-Wise Step Size")
ax1.set_ylabel("Step Size")
# ax1.set_ylim(top=10)
# ax1.set_yscale('log')

ax2.plot(debug_hist.L.T)
ax2.set_title("Chain-Wise L")
ax2.set_ylabel("L")
# ax2.set_ylim(top=20)



inverse_mass_matrix_hist = debug_hist.inverse_mass_matrix[0]
vmapped_eigval = jax.vmap(lambda x: jnp.linalg.eig(x)[0])
mass_mat_eigval = vmapped_eigval(inverse_mass_matrix_hist)
min_eigval = jnp.min(mass_mat_eigval, axis=1)
max_eigval = jnp.max(mass_mat_eigval, axis=1)
mean_eigval = jnp.mean(mass_mat_eigval, axis=1)

final_eigvals = jnp.real(jnp.linalg.eig(final_cov)[0])
min_eigval_final = jnp.min(final_eigvals)
max_eigval_final = jnp.max(final_eigvals)
mean_eigval_final = jnp.mean(final_eigvals)

ax3.plot(min_eigval, label='Min', color='blue')
ax3.axhline(min_eigval_final, color='blue', linestyle='--')
ax3.plot(mean_eigval, label='Mean', color='black')
ax3.axhline(mean_eigval_final, color='black', linestyle='--')
ax3.plot(max_eigval, label='Max', color='red')
ax3.axhline(max_eigval_final, color='red', linestyle='--')



ax3.legend()
ax3.set_title("Covariance Eigenvalues")
ax3.set_yscale('log')
ax3.set_ylabel("Eigenvalue")



smooth_kernel_size = 30
kernel = np.ones(smooth_kernel_size) / smooth_kernel_size
xi_chain = debug_hist.xi[8]
xi_smoothed = np.convolve(xi_chain, kernel, mode='same')
ax4.plot(xi_chain, alpha=0.5, color='blue')
ax4.plot(xi_smoothed, alpha=1.0, color='blue')
ax4.set_yscale('log')
ax4.set_ylabel("xi for chain 0")
ax4.axhline(1.0, color='black', linestyle='--')

ax_last.set_xlabel("Step")

ax_last.set_title("Nans?")
ax_last.imshow(debug_hist.nonan[:,:stage2], aspect='auto', interpolation='none', cmap='RdYlGn')

for ax in axs:
    ax.axvline(stage1, color='red', linestyle='--')
    ax.axvline(stage2, color='blue', linestyle='--')
    ax.axvline(stage3, color='green', linestyle='--')

    # ax.set_xlim(right=stage3)


plt.show()

In [ ]:
from alternate_inference import laps_blackjax
importlib.reload(laps_blackjax)
from alternate_inference import laps_blackjax
result = laps_blackjax.LAPS_blackjax(
    model_seq,
    qz,
    n_hmc=128,
    num_unadjusted_steps=1000,
    num_adjusted_steps=2000,
    seed=0,
)

# import laps
# importlib.reload(laps)
# from laps import LAPS_JIT
# debug_hist, carry = LAPS_JIT(
#     model_seq,
#     qz,
#     n_hmc=128,
#     num_unadjusted_steps=100,
#     num_adjusted_steps=100,
#     num_results=500,
#     adapt_mass_matrix=True,
#     debug_output=True,
#     alpha=jnp.float32(20.0),
#     seed=0,
#     # switch_threshold=jnp.float32(0.01),
#     # mass_matrix_trigger_threshold=jnp.float32(0.2),
# )

In [ ]:
laps_samples =result['final_state'].position #result["info"]["phase_2"]
#result['final_state'].position#.shape

In [ ]:
result["info"]["phase_2"]

In [ ]:
# laps_samples = debug_hist[2].position
# import laps
# importlib.reload(laps)
# from laps import plot_laps_diagnostics

# fig, axs = plot_laps_diagnostics(debug_hist, carry=carry)
# plt.show()

In [ ]:
print(jnp.max(blackjax.diagnostics.potential_scale_reduction(mclmc_samples, chain_axis=0, sample_axis=1)))
print(blackjax.diagnostics.effective_sample_size(mclmc_samples, chain_axis=0, sample_axis=1))

print(jnp.max(blackjax.diagnostics.potential_scale_reduction(laps_samples, chain_axis=0, sample_axis=1)))
print(blackjax.diagnostics.effective_sample_size(laps_samples, chain_axis=0, sample_axis=1))

In [ ]:
print(jnp.max(blackjax.diagnostics.potential_scale_reduction(hmc_samples, chain_axis=0, sample_axis=1)))
print(blackjax.diagnostics.effective_sample_size(hmc_samples, chain_axis=0, sample_axis=1))

In [ ]:
dim = mclmc_samples.shape[-1]
run_key = jax.random.key(0)
samples = mclmc_samples.reshape(-1, dim) #samples_mclmc 
MCMC_x = prob_model.bij.forward(list(samples.T))

laps_smp = laps_samples.reshape(-1, dim)
LAPS_x = prob_model.bij.forward(list(laps_smp.T))

# adapt_cov = carry.inverse_mass_matrix#debug_hist.inverse_mass_matrix[0,-1]
# adapt_qz = tfd.MultivariateNormalFullCovariance(loc=jnp.mean(laps_smp, axis=0), covariance_matrix=adapt_cov)
# adapt_qz_samples = adapt_qz.sample((10000,), run_key)
# adapt_qz_x = prob_model.bij.forward(list(adapt_qz_samples.T))

SVI_samples = qz.sample((1000,), run_key)
SVI_x = prob_model.bij.forward(list(SVI_samples.T))

# HMC_1chain_x = prob_model.bij.forward(list(results['HMC'].HMC_samples_z[0, 11, -40000:].T))

plot_params = cornerplot_labels(MCMC_x)

hmc_x = prob_model.bij.forward(list(hmc_samples.reshape(-1, dim).T))

n_samp = hmc_samples.shape[0]*hmc_samples.shape[1]
rand_idx = np.random.choice(np.arange(n_samp), size=(10000,), replace=True)

fig = cornerplot_posterior(jax.tree.map(lambda x: x[rand_idx], hmc_x), color='black',plot_params=plot_params)

n_samp_MCMC = MCMC_x[0][0]['e1'].shape[0]
rand_idx_MCMC = np.random.choice(np.arange(n_samp_MCMC), size=(10000,), replace=True)
# cornerplot_posterior(jax.tree.map(lambda x: x[rand_idx_MCMC], MCMC_x), fig=fig, color='red',plot_params=plot_params)


n_samp_laps = LAPS_x[0][0]['e1'].shape[0]
rand_idx_laps = np.random.choice(np.arange(n_samp_laps), size=(10000,), replace=True)
cornerplot_posterior(jax.tree.map(lambda x: x[rand_idx_laps], LAPS_x), fig=fig, color='orange',plot_params=plot_params)


cornerplot_posterior(SVI_x, fig=fig, color='blue', plot_params=plot_params)
# cornerplot_posterior(adapt_qz_x, fig=fig, color='purple')
# cornerplot_posterior(mclmc_results_100sys.HMC_samples, fig=fig, color='purple', plot_params=plot_params)
# cornerplot_posterior(HMC_1chain_x, fig=fig, color='green', plot_params=plot_params)

plt.show()